# Predicting failure within 90 days of an inspection
Target: `failed_90d`. The model is meant for future inspections, so the split is chronological.

Decisions up front:
- `asset_tag` is an identifier: dropped. (Each tag appears once, so there is no entity overlap to guard against; verified below.)
- `parts_replaced` is recorded on the work order that *closes* the inspection record, i.e. after the inspection and likely after the failure/repair. It is not known at prediction time and is dropped. (Its near-perfect relation with the target below confirms leakage.)
- `inspected_at` is used only for ordering, not as a feature.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
from sklearn.base import clone
SEED=0
df=pd.read_csv('data.csv',parse_dates=['inspected_at']).sort_values('inspected_at',kind='stable').reset_index(drop=True)
print(df.shape, 'unique assets:',df.asset_tag.nunique(), 'failure rate: %.3f'%df.failed_90d.mean())
print(df.groupby('parts_replaced').failed_90d.agg(['mean','size']))

## 1. Test set: the latest 20% of inspections, set aside before any modelling

In [ ]:
cut=int(len(df)*0.8)
train,test=df.iloc[:cut].copy(),df.iloc[cut:].copy()
assert train.inspected_at.max()<=test.inspected_at.min()
assert not set(train.asset_tag)&set(test.asset_tag)
print(len(train),len(test),train.inspected_at.max(),test.inspected_at.min())
FEATURES=['site','vibration_rms','bearing_temp_c','ambient_temp_c','meter_hours','load_pct']
NUM=[f for f in FEATURES if f!='site']
Xtr,ytr=train[FEATURES],train.failed_90d.values
Xte,yte=test[FEATURES],test.failed_90d.values

## 2-4. Candidate models (named before any scoring) and CV scheme
Candidates:
1. `baseline`: predicts the training prior (trivial).
2. `logreg_C0.1`, `logreg_C1`, `logreg_C10`: logistic regression, standardised numerics + one-hot site.
3. `hgb_shallow`, `hgb_deep`: histogram gradient boosting (depth 3 / lr 0.05, depth 6 / lr 0.05).

CV: 5-fold expanding-window `TimeSeriesSplit` on the training rows only (earlier train, later validate). All preprocessing sits inside the pipeline, so it is fit per fold. Selection metric: mean fold average precision (PR-AUC; classes are imbalanced). Threshold: maximise F1 on pooled out-of-fold predictions of the chosen model.

In [ ]:
def prep(): return ColumnTransformer([('n',Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]),NUM),
                                      ('c',OneHotEncoder(handle_unknown='ignore'),['site'])])
CANDIDATES={
 'baseline':Pipeline([('p',prep()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=2000))]),
 'logreg_C1':Pipeline([('p',prep()),('m',LogisticRegression(C=1,max_iter=2000))]),
 'logreg_C10':Pipeline([('p',prep()),('m',LogisticRegression(C=10,max_iter=2000))]),
 'hgb_shallow':Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=SEED))]),
 'hgb_deep':Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,max_iter=200,random_state=SEED))]),
}
cv=TimeSeriesSplit(n_splits=5)
rows,oof={}, {}
for name,mdl in CANDIDATES.items():
    ap,auc,idx,pr=[],[],[],[]
    for a,b in cv.split(Xtr):
        m=clone(mdl).fit(Xtr.iloc[a],ytr[a]); p=m.predict_proba(Xtr.iloc[b])[:,1]
        ap.append(average_precision_score(ytr[b],p)); auc.append(roc_auc_score(ytr[b],p)); idx.append(b); pr.append(p)
    oof[name]=(np.concatenate(idx),np.concatenate(pr))
    rows[name]=dict(cv_ap=np.mean(ap),cv_ap_sd=np.std(ap),cv_auc=np.mean(auc))
cvres=pd.DataFrame(rows).T.sort_values('cv_ap',ascending=False); cvres

## 5. Choose model and threshold on CV only

In [ ]:
best=cvres.index[0]
idx,p=oof[best]; y=ytr[idx]
grid=np.linspace(0.02,0.9,89)
f1s=[f1_score(y,p>=t,zero_division=0) for t in grid]
THRESH=float(grid[int(np.argmax(f1s))])
print('chosen model:',best,'| threshold: %.2f'%THRESH,'| CV F1 at threshold: %.3f'%max(f1s))
print('CV precision %.3f recall %.3f'%(precision_score(y,p>=THRESH),recall_score(y,p>=THRESH)))

**Recorded choice (made from CV results above, before the test set is touched):** the model, feature set (`site` + five measurements; no `parts_replaced`, no identifiers) and threshold printed in the previous cell are final. No further changes after viewing test results.

## 6. Refit on all training rows; score the test set once (headline)

In [ ]:
final=clone(CANDIDATES[best]).fit(Xtr,ytr)
pt=final.predict_proba(Xte)[:,1]
base=clone(CANDIDATES['baseline']).fit(Xtr,ytr).predict_proba(Xte)[:,1]
print('TEST  ROC-AUC %.3f | PR-AUC %.3f (prevalence %.3f)'%(roc_auc_score(yte,pt),average_precision_score(yte,pt),yte.mean()))
print('TEST  at threshold %.2f: precision %.3f recall %.3f F1 %.3f'%(THRESH,precision_score(yte,pt>=THRESH,zero_division=0),recall_score(yte,pt>=THRESH),f1_score(yte,pt>=THRESH)))
print('baseline ROC-AUC %.3f PR-AUC %.3f'%(roc_auc_score(yte,base),average_precision_score(yte,base)))

## Conclusion
See the printed test metrics above; they are the headline and were computed once. Caveat: results hold for future inspections of assets not seen in training and assume the same sensor and operating regime.